# 1. Données et graphe de connaissances

Corpus EUR-Lex (100 actes du numérique), relations entre actes et renvois entre articles, graphe Neo4j et contrôle manuel de la qualité des liens extraits.

_Notebook rejoué avec le code du projet (`graphrag/`, et `app/` du projet vectoriel). Ses sorties sont réelles : il a été exécuté le 10/11/2026 à 03:04 par `scripts/build_notebooks.py`. Aucun appel au LLM n'est fait ici ; Neo4j doit être démarré._

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# candidats de recherche déjà calculés (MultiQuery, HyDE) : aucune requête au LLM
os.environ.setdefault("SEED_CACHE_PATH", str(ROOT / "data" / "cache" / "user_test_candidates.json"))

import pandas as pd
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)

from graphrag import config
config.use_vector_rag()
DATA = config.VECTOR_RAG_DIR / "data" / "eurlex"
RESULTS = ROOT / "results"
print("Projet :", ROOT)
print("Données du projet vectoriel :", DATA)

Projet : C:\Users\hazem\5eme\GraphRAG-juridique
Données du projet vectoriel : C:\Users\hazem\5eme\Advenced-Rag-Agents-main\Advenced-Rag-Agents-main\data\eurlex


## 1.1 Le corpus

Collecté par `collect_eurlex.py` (projet vectoriel) : textes français par l'API REST Cellar, métadonnées et relations par SPARQL. L'unité de travail est l'article.

In [2]:
articles = pd.read_json(DATA / "articles.jsonl", lines=True)
documents = pd.read_csv(DATA / "documents.csv", dtype=str)
print(len(documents), "actes ;", len(articles), "articles")
articles["doc_type"].value_counts().rename("articles par type d'acte")

100 actes ; 3941 articles


,articles par type d'acte
doc_type,
REG,3143
DIR,782
DEC,16


In [3]:
longueur = articles["text"].str.len()
longueur.describe(percentiles=[0.5, 0.9]).round(0).rename("longueur d'un article (caractères)")

,longueur d'un article (caractères)
count,3941.0
mean,2217.0
std,5391.0
min,48.0
50%,1452.0
90%,4399.0
max,219254.0


In [4]:
relations = pd.read_csv(DATA / "relations.csv", dtype=str)
refs = pd.read_csv(DATA / "article_refs.csv", dtype=str)
print("Relations acte -> acte dans le corpus (Cellar) :", relations["type"].value_counts().to_dict())
print("Renvois article -> article extraits par regex :", len(refs))
print("  dont vers un article du corpus :", int((refs["target_in_corpus"] == "True").sum()))

Relations acte -> acte dans le corpus (Cellar) : {'CITES': 619, 'AMENDS': 18, 'BASED_ON': 2}
Renvois article -> article extraits par regex : 8023
  dont vers un article du corpus : 7047


## 1.2 Le graphe Neo4j

Construit par `graphrag/build_graph.py`. Comptes lus directement dans la base.

In [5]:
from graphrag.db import run

pd.DataFrame(run("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS noeuds ORDER BY noeuds DESC"))

,label,noeuds
0,Article,3941
1,Document,768
2,EuroVoc,354
3,Cluster,25


In [6]:
pd.DataFrame(run("""MATCH (a)-[r]->(b)
    RETURN type(r) AS arete, labels(a)[0] AS de, labels(b)[0] AS vers, count(*) AS n ORDER BY n DESC"""))

,arete,de,vers,n
0,REFERS_TO,Article,Article,7047
1,HAS_ARTICLE,Document,Article,3941
2,IN_CLUSTER,Article,Cluster,3941
3,NEXT,Article,Article,3841
4,MENTIONS,Article,Document,1525
5,HAS_SUBJECT,Document,EuroVoc,861
6,CITES,Document,Document,619
7,BASED_ON,Document,Document,495
8,AMENDS,Document,Document,175
9,AMENDS,Article,Document,109


In [7]:
inter = run("MATCH (a:Article)-[:REFERS_TO]->(b:Article) WHERE a.celex <> b.celex RETURN count(*) AS n")[0]["n"]
print("Renvois REFERS_TO entre actes différents :", inter)

Renvois REFERS_TO entre actes différents : 629


### Deux exemples de chemins utilisés par le Graph RAG

In [8]:
# Quel article modifie le règlement 2019/1020 ? (arête Article -> Document)
pd.DataFrame(run("""MATCH (a:Article)-[r:AMENDS]->(d:Document {celex: '32019R1020'})
    RETURN a.id AS article, a.title AS titre, r.evidence AS preuve"""))

,article,titre,preuve
0,32024R2847_art_66,Modification du règlement (UE) 2019/1020,titre


In [9]:
# L'article 28 du DSA (mineurs) renvoie à l'article 4 du RGPD (définition du profilage)
pd.DataFrame(run("""MATCH (a:Article {id: '32022R2065_art_28'})-[:REFERS_TO]->(b:Article)
    RETURN a.title AS de, b.id AS vers, b.title AS titre"""))

,de,vers,titre
0,Protection des mineurs en ligne,32016R0679_art_4,Définitions


## 1.3 Contrôle qualité des liens extraits par règles

50 arêtes tirées au hasard (graine 42) par `graphrag/sample_edges.py`, puis annotées à la main dans `results/edge_sample.csv` (colonne `correct`).

In [10]:
sample = pd.read_csv(RESULTS / "edge_sample.csv", sep=";")
qualite = sample.groupby("kind")["correct"].agg(corrects="sum", total="count")
qualite["precision"] = (qualite["corrects"] / qualite["total"]).round(3)
print("Total :", int(sample["correct"].sum()), "/", len(sample), "=", round(sample["correct"].mean(), 3))
qualite

Total : 47 / 50 = 0.94


,corrects,total,precision
kind,,,
AMENDS (phrase),3,3,1.000
AMENDS (phrase_cellar),2,2,1.000
AMENDS (titre),6,6,1.000
REFERS_TO,28,30,0.933
REPEALS (phrase),5,6,0.833
REPEALS (phrase_cellar),1,1,1.000
REPEALS (titre),2,2,1.000


In [11]:
sample.loc[sample["correct"] == 0, ["kind", "source", "target", "erreur"]]

,kind,source,target,erreur
14,REFERS_TO,32024R1358_art_2,32024R1358_art_20,"mauvais acte (renvoi vers le règlement 2024/1347, rattaché au 2024/1358)"
25,REFERS_TO,32018R1046_art_275,32018R1046_art_22,"article d un acte modifié (art. 22 du règlement 1316/2013, pas du 2018/1046)"
30,REPEALS (phrase),32018R1046_art_281,32012R1268,abrogation indirecte (la Commission abroge le règlement délégué 1268/2012)
